# Suite LOG — The logical layout

`Logical` defines a diagram's logical layout: diagrams, shapes, points, vectors, segments and adjacency specs, each a reference object owned through `Logical.Owns`, and each container's grid, which holds its children, each playing its part through a relation of its own, with no coordinates; and resolves the compass of a point or a shape: mbse-basis [CAP-237: logical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-237.md), and [CAP-280: the grid in the logical layout](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-280), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders, Paths, Validators
from mbse.Diagrams.Framework import Logical
from support import text

store = Builders.Store()
store.transaction(Logical.install)
L = store.Logical
directory = store.singleton("Logical.Diagrams")
problems = Validators.Validate(store).Reachable


def owned(owner, name):
    """The entry by which `owner` owns what it builds, as `name`."""
    return lambda a: a.insert(lambda e: e.name(name).owner(owner))


def diagram(name):
    return L.OfDiagram(lambda d: d.owner(owned(directory, name)).directory(lambda a: a.insert(lambda e: e.directory(directory))))


def grid(container):
    """The grid `container`, a shape or the diagram, owns, made where it owns none."""
    return Logical.grid_of(container) or L.OfGrid(lambda g: g.owner(owned(container, "grid")))


def shape(diagram, name, container=None):
    """A shape owned by `diagram` and a member of it, and held by the grid of `container`, a shape, where one is given:
    else at the top level, which the diagram's grid holds."""
    def spec(s):
        s = s.owner(owned(diagram, name)).diagram(lambda a: a.insert(lambda e: e.diagram(diagram)))
        return s if container is None else s.container(lambda a: a.insert(lambda e: e.grid(grid(container))))
    return L.OfShape(spec)


def point(of, name, role):
    """A point owned by the shape `of`, as its origin, a border or an extent point."""
    link = {"origin": "origin_of", "border": "border_of", "extent": "extent_of"}[role]
    return L.OfPoint(lambda p: getattr(p.owner(owned(of, name)), link)(lambda a: a.insert(lambda e: e.shape(of))))


def vector(of, name, start, end, degrees=None, length=1.0):
    def spec(v):
        v = v.owner(owned(of, name)).start(lambda a: a.insert(lambda e: e.point(start)))
        v = v.end(lambda a: a.insert(lambda e: e.point(end))).length(length)
        return v.rotation(lambda r: r.degrees(degrees)) if degrees is not None else v
    return L.OfVector(spec)


def raises(message, block):
    try:
        with store.transaction():
            block()
    except ValueError as error:
        assert str(error) == message, str(error)
    else:
        raise AssertionError(f"expected ValueError: {message}")


path = lambda x: JSON.ToJSON(store)(store.Paths.OfRoot.as_schema(), Paths.of(x))
names = lambda owner: [e.name.get() for e in owner.owns]

<a id="LOG-01"></a>
## LOG-01 · The install defines the logical layout's schemas, each valid, and run again writes nothing

In [ ]:
NAMES = ['Logical.Owns', 'Logical.OfRotation', 'Logical.DiagramDirectory', 'Logical.InDiagrams', 'Logical.OfDiagram', 'Logical.InDiagram', 'Logical.OfShape', 'Logical.OfGrid', 'Logical.Containment', 'Logical.OfPoint', 'Logical.Origin', 'Logical.Borders', 'Logical.Extents', 'Logical.OfVector', 'Logical.VectorStart', 'Logical.VectorEnd', 'Logical.OfSegment', 'Logical.SegmentEnds', 'Logical.OfAdjacency', 'Logical.AdjacencyFrom', 'Logical.AdjacencyTo']
schemas = [store.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
store.transaction(Logical.install)
assert [store.registered(name) for name in NAMES] == schemas and directory is store.Instances.Logical.Diagrams

<a id="LOG-02"></a>
## LOG-02 · Everything is owned through Logical.Owns, by a name unique among its owner's, its path read back through it: every shape by its diagram, contained or not, and a container's grid by the container

In [ ]:
with store.transaction():
    overview = diagram("overview")
    cpu = shape(overview, "cpu")
    alu = shape(overview, "alu", cpu)
    o, ne, e = point(cpu, "origin", "origin"), point(cpu, "ne", "border"), point(cpu, "e", "extent")
    vector(cpu, "to ne", o, ne, 45.0, 2.0)
    vector(cpu, "to e", o, e, 90.0)
    point(alu, "origin", "origin")
assert text([names(directory), names(overview), names(cpu), names(alu)]) == '[["overview"],["cpu","alu"],["grid","origin","ne","e","to ne","to e"],["origin"]]'
assert text([Logical.owner_of(x) is owner for x, owner in ((overview, directory), (cpu, overview), (alu, overview), (o, cpu), (Logical.grid_of(cpu), cpu))]) == '[true,true,true,true,true]'
assert text(path(alu.owns({"name": "origin"}))) == '"{\\"singleton\\": \\"Logical.Diagrams\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"overview\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"alu\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"origin\\"}, \\"next\\": {\\"target\\": {}}}}}}}}}"'
assert text([Logical.owner_of(directory), Logical.name_of(directory), Logical.name_of(cpu)]) == '[null,null,"cpu"]'
assert text(problems(L.OfDiagram, overview)) == '[]'

<a id="LOG-03"></a>
## LOG-03 · A shape's roles: a member of its diagram, contained or not, and held by its container's grid where it has one, one origin, and border and extent points; refused without an owner

In [ ]:
assert text([[Logical.name_of(x.shape) for x in overview.shapes], [x.child is alu for x in Logical.grid_of(cpu).children], [x.grid is Logical.grid_of(cpu) for x in alu.container], list(cpu.container)]) == '[["cpu","alu"],[true],[true],[]]'
assert text([Logical.container_of(alu) is cpu, Logical.container_of(cpu) is overview, Logical.grid_of(alu), Logical.holding(alu) is Logical.grid_of(cpu), Logical.holding(cpu)]) == '[true,true,null,true,null]'
with store.transaction():
    stray = L.OfShape(lambda s: s.owner(owned(overview, "stray")))  # no member of the diagram
assert text([Logical.container_of(stray), Logical.holding(stray)]) == '[null,null]'
with store.transaction():
    L.OfDiagram(overview, lambda d: d.owns(lambda a: a.remove(lambda e: e.name("stray"))))
assert text([[x.point is o for x in cpu.origin], [x.point is ne for x in cpu.borders], [x.point is e for x in cpu.extents]]) == '[[true],[true],[true]]'
with store.transaction():
    point(cpu, "second", "origin")
assert text(problems(L.OfShape, cpu)) == '["unique(point) violated: entries agreeing on [\'shape\'] differ on [\'point\']"]'
with store.transaction():
    L.OfShape(cpu, lambda s: s.owns(lambda a: a.remove(lambda e: e.name("second"))))
assert text([problems(L.OfShape, cpu), names(cpu)]) == '[[],["grid","origin","ne","e","to ne","to e"]]'
seen = []  # read before the commit refuses it
raises("a 'Logical.OfPoint' needs an owner through 'owner': give it one, since no singleton can be",
       lambda: seen.extend((lambda q: [Logical.owner_of(q), Logical.name_of(q)])(L.OfPoint(lambda p: p))))
assert text(seen) == '[null,null]'

<a id="LOG-04"></a>
## LOG-04 · A vector runs from one start to one end, by a rotation and a length

In [ ]:
v = cpu.owns({"name": "to ne"})
assert text([Logical.ends(v)[0] is o, Logical.ends(v)[1] is ne, v.rotation.degrees.get(), v.length.get()]) == '[true,true,45.0,2.0]'
assert text([[x.vector is v for x in ne.incoming], len(list(o.outgoing))]) == '[[true],2]'
with store.transaction():
    L.OfVector(v, lambda x: x.end(lambda a: a.insert(lambda e: e.point(o))))
assert text(problems(L.OfVector, v)) == '["unique(point) violated: entries agreeing on [\'vector\'] differ on [\'point\']"]'
with store.transaction():
    L.OfVector(v, lambda x: x.end(lambda a: a.remove(lambda e: e.point(o))))
assert text(problems(L.OfVector, v)) == '[]'

<a id="LOG-05"></a>
## LOG-05 · A segment joins two points, with no length or rotation; an adjacency spec runs from one shape to another

In [ ]:
with store.transaction():
    seg = L.OfSegment(lambda s: s.owner(owned(cpu, "edge")).ends(lambda a: a.insert(lambda x: x.point(ne)).insert(
        lambda x: x.point(e))))
    fpu = shape(overview, "fpu", cpu)
    spec = L.OfAdjacency(lambda s: s.owner(owned(alu, "to fpu 90")).from_shape(lambda a: a.insert(lambda x: x.shape(alu)))
                         .to_shape(lambda a: a.insert(lambda x: x.shape(fpu))).rotation(lambda r: r.degrees(90.0)))
assert text([[x.point is p for x, p in zip(seg.ends, (ne, e))], [x.segment is seg for x in e.segments]]) == '[[true,true],[true]]'
assert text([[x.adjacency is spec for x in alu.adjacent_to], [x.adjacency is spec for x in fpu.adjacent_from], spec.rotation.degrees.get(), names(alu)]) == '[[true],[true],90.0,["origin","to fpu 90"]]'
assert text(problems(L.OfDiagram, overview)) == '[]'

<a id="LOG-06"></a>
## LOG-06 · A compass: a shape's and its origin's north, a border or extent point's its vector's rotation

In [ ]:
with store.transaction():
    sw = point(cpu, "sw", "extent")
    vector(cpu, "to sw", o, sw, 585.5)
    w = point(cpu, "w", "border")
    vector(cpu, "to w", o, w, -90.0)
assert text([Logical.compass(x) for x in (cpu, o, alu, ne, e, sw, w)]) == '[0.0,0.0,0.0,45.0,90.0,225.5,270.0]'

<a id="LOG-07"></a>
## LOG-07 · A compass the rules leave undefined is refused

In [ ]:
with store.transaction():
    bare = shape(overview, "bare", cpu)
    x = point(bare, "x", "border")
    q = point(fpu, "q", "border")
    point(fpu, "origin", "origin")
    s = point(cpu, "s", "border")
    vector(cpu, "to s", o, s)
    t = point(cpu, "t", "border")
    vector(cpu, "to t 1", o, t, 1.0)
    vector(cpu, "to t 2", o, t, 2.0)
raises("a shape without an origin has no compass", lambda: Logical.compass(bare))
raises("a shape without an origin has no compass", lambda: Logical.compass(x))
raises("a point with 0 vectors from its shape's origin, not one, has no compass", lambda: Logical.compass(q))
raises("a point with 2 vectors from its shape's origin, not one, has no compass", lambda: Logical.compass(t))
raises("a point whose vector from its shape's origin has no rotation has no compass", lambda: Logical.compass(s))

<a id="LOG-08"></a>
## LOG-08 · A grid's settings: its own where valid, else its diagram's, else 3 and rows; its cell rotation turns the compass of each shape it holds; the diagram's holds each top-level shape, with no entry

In [ ]:
g = Logical.grid_of(cpu)
assert text([Logical.settings(g), Logical.settings(Logical.grid_of(alu) or g)]) == '[[3,"rows"],[3,"rows"]]'
with store.transaction():
    L.OfDiagram(overview, lambda d: d.grid_span(2).grid_grows("columns"))
assert text(Logical.settings(g)) == '[2,"columns"]'
with store.transaction():
    L.OfGrid(g, lambda x: x.span(4).grows("rows"))
assert text(Logical.settings(g)) == '[4,"rows"]'
with store.transaction():
    L.OfGrid(g, lambda x: x.span(0).grows("diagonal"))
assert text(Logical.settings(g)) == '[2,"columns"]'
with store.transaction():
    L.OfDiagram(overview, lambda d: d.grid_span(-1).grid_grows("diagonal"))
assert text(Logical.settings(g)) == '[3,"rows"]'
with store.transaction():
    L.OfGrid(g, lambda x: x.cell_rotation(lambda r: r.degrees(90.0)))
    top = grid(overview)
    L.OfGrid(top, lambda x: x.cell_rotation(lambda r: r.degrees(180.0)))
assert text([Logical.compass(x) for x in (cpu, alu, e)]) == '[180.0,270.0,270.0]'
assert text([Logical.holding(cpu) is top, list(top.children), problems(L.OfDiagram, overview)]) == '[true,[],[]]'